# Lab Assignment 3 — Task 1

## Part 1: Parsing a Sentence with CKYGiven the following grammar in CNF.1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.2. Also print the final CKY chart/table.

In [29]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG

In [30]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")
sentence = "the man saw the dog"

In [31]:
'''Now that the grammar is defined, implement the CKY algorithm here'''# same grammar as above, but written as plain python rules so we can# run our own CKY without depending on nltk's CFG objectbinary_rules = [    ("S", "NP", "VP"),    ("NP", "Det", "N"),    ("VP", "V", "NP"),]lexical_rules = {    "the": ["Det"],    "man": ["N"],    "dog": ["N"],    "saw": ["V"],}def cky(words):    n = len(words)    # chart[i][j] holds every category that can generate words[i:j]    # (0-indexed, j exclusive), keyed as a dict of dicts instead of a    # flat 2D list just to show a different layout than usual    chart = {i: {j: set() for j in range(n + 1)} for i in range(n)}    # fill the diagonal: single words get their POS tags    for i, w in enumerate(words):        chart[i][i + 1] |= set(lexical_rules.get(w, []))    # fill everything above the diagonal, span length 2 upwards    for span in range(2, n + 1):        for i in range(0, n - span + 1):            j = i + span            found = set()            for split in range(i + 1, j):                left = chart[i][split]                right = chart[split][j]                for lhs, b, c in binary_rules:                    if b in left and c in right:                        found.add(lhs)            chart[i][j] |= found    return chart, nwords = sentence.split()chart, n = cky(words)can_parse = "S" in chart[0][n]print(f"Sentence: '{sentence}'")print("Can be parsed?", can_parse)print("\nCKY chart (row i, col j -> categories spanning words[i:j]):")for i in range(n):    row = []    for j in range(n + 1):        cats = chart[i][j]        row.append("{" + ",".join(sorted(cats)) + "}" if cats else "-")    print(f"i={i}: " + "  ".join(row))

'Now that the grammar is defined, implement the CKY algorithm here'

## Part 2: Comparision with NLTK's chart parserAfter implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

In [32]:
# run nltk's own chart parser on the same grammar + sentenceparser = ChartParser(cky_grammar)nltk_trees = list(parser.parse(words))print("Our CKY says parsable:", can_parse)print("NLTK found", len(nltk_trees), "parse tree(s)")for t in nltk_trees:    print(t)# the two should always agree: nltk finds a tree exactly when# our chart has S in chart[0][n]agree = can_parse == (len(nltk_trees) > 0)print("\nResults match:", agree)

---

# Lab Assignment 3 — Task 2

# Lab Assignment 3 — Task 2## Converting a CFG to Chomsky Normal Form (CNF)A grammar is in CNF if every rule looks like:1. `A -> B C` (two non-terminals), or2. `A -> 'word'` (one terminal).

## PART 1 : The Challenge of Ambiguity & CNF Constraints> **I saw the man with a telescope.**Two readings:1. I saw a man who was holding a telescope.2. I used a telescope to see the man.### Task1. Convert the PCFG to strict CNF (find the rule that breaks CNF, split it into binary rules, keep the same probability).2. Parse the sentence with `ViterbiParser`, print the best tree and its probability.3. Explain why that reading won, using the actual rule probabilities.

In [33]:
pcfg_grammar_str = """    S -> NP VP [1.0]    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]    VP -> V NP [0.7] | V NP PP [0.3]    PP -> P NP [1.0]    Det -> 'the' [0.8] | 'a' [0.2]    N -> 'man' [0.5] | 'telescope' [0.5]    V -> 'saw' [1.0]    P -> 'with' [1.0]"""
sentence = "I saw the man with a telescope"

In [34]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser, pchart
# the only rule that isn't binary is:  VP -> V NP PP [0.3]
# (it has 3 symbols on the right, CNF only allows 1 or 2)
# fix: introduce a helper non-terminal that groups "NP PP" together,
# and give it probability 1.0 so the overall path keeps probability 0.3

#   VP -> V NP_PP   [0.3]
#   NP_PP -> NP PP  [1.0]
cnf_grammar_str = """    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP_PP [0.3]
    NP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]"""
cnf_pcfg = PCFG.fromstring(cnf_grammar_str)
# quick check that every rule really is A->BC or A->'word'
for rule in cnf_pcfg.productions():
    assert len(rule.rhs()) <= 2, f"not CNF: {rule}"
print("grammar checked, every rule has at most 2 symbols on the right")

grammar checked, every rule has at most 2 symbols on the right


In [35]:
tokens = sentence.split()
viterbi = ViterbiParser(cnf_pcfg)
best_tree = list(viterbi.parse(tokens))[0]
print("Best parse tree:")
print(best_tree)
print(f"\nProbability: {best_tree.prob():.10f}")

Best parse tree:
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Probability: 0.0001512000


### AnalysisInstead of typing the numbers in by hand, pull every valid parse out of the grammar, sort them by probability, and read the winning rules straight off the tree.

In [36]:
# InsideChartParser gives us every valid parse (not just the best one),# so we can actually compare the two competing readingschart_parser = pchart.InsideChartParser(cnf_pcfg)all_parses = sorted(chart_parser.parse(tokens), key=lambda t: t.prob(), reverse=True)print(f"found {len(all_parses)} valid parses\n")for rank, tree in enumerate(all_parses, start=1):    print(f"#{rank}  p={tree.prob():.6f}")    print(tree, "\n")winner, runner_up = all_parses[0], all_parses[1]# map (lhs, rhs) -> probability so we can print the exact rules each tree usedprobs = {}for rule in cnf_pcfg.productions():    lhs = rule.lhs().symbol()    rhs = tuple(s.symbol() if hasattr(s, "symbol") else s for s in rule.rhs())    probs[(lhs, rhs)] = rule.prob()def show_rules(tree, label):    print(f"{label} (p={tree.prob():.6f}):")    for rule in tree.productions():        lhs = rule.lhs().symbol()        rhs = tuple(s.symbol() if hasattr(s, "symbol") else s for s in rule.rhs())        p = probs.get((lhs, rhs))        print(f"  {lhs} -> {' '.join(rhs)}   [{p}]")    print()show_rules(winner, "Winning reading (man holds the telescope)")show_rules(runner_up, "Other reading (telescope used to see)")print(f"winner / runner-up = {winner.prob() / runner_up.prob():.4f}")print(    "\nBoth trees share the same word-level rules (Det/N/V/P), so it comes down to:\n"    "  NP -> NP PP   [0.6]   x  VP -> V NP  [0.7]   = 0.42   (winner, PP attaches to NP)\n"    "  VP -> V NP_PP [0.3]   x  NP_PP -> NP PP [1.0] = 0.30  (runner-up, PP attaches to VP)\n"    "0.42 > 0.30, so ViterbiParser picks the NP-attachment reading — the man is holding the telescope.")

# Part 2: Advanced Dependency Parsing Using spaCy> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**

## 1. Initialize and ParseLoad `en_core_web_sm` and process the sentence into `parsed_doc`.

In [37]:
import spacy
nlp = spacy.load("en_core_web_sm")
sentence2 = ("The exhausted researchers at the institute discovered a new "             "vaccine that completely prevents the viral mutation.")
parsed_doc = nlp(sentence2)
for tok in parsed_doc:
    print(f"{tok.i:2d}  {tok.text:12s} pos={tok.pos_:6s} dep={tok.dep_:8s} head={tok.head.text}")

 0  The          pos=DET    dep=det      head=researchers
 1  exhausted    pos=ADJ    dep=amod     head=researchers
 2  researchers  pos=NOUN   dep=nsubj    head=discovered
 3  at           pos=ADP    dep=prep     head=researchers
 4  the          pos=DET    dep=det      head=institute
 5  institute    pos=NOUN   dep=pobj     head=at
 6  discovered   pos=VERB   dep=ROOT     head=discovered
 7  a            pos=DET    dep=det      head=vaccine
 8  new          pos=ADJ    dep=amod     head=vaccine
 9  vaccine      pos=NOUN   dep=dobj     head=discovered
10  that         pos=PRON   dep=nsubj    head=prevents
11  completely   pos=ADV    dep=advmod   head=prevents
12  prevents     pos=VERB   dep=relcl    head=vaccine
13  the          pos=DET    dep=det      head=mutation
14  viral        pos=ADJ    dep=amod     head=mutation
15  mutation     pos=NOUN   dep=dobj     head=prevents
16  .            pos=PUNCT  dep=punct    head=discovered


## 2. Isolate the RootFind the token with `dep_ == "ROOT"` and save it as `root_node`.

In [38]:
def find_root(doc):
    # walk the doc once, grab the token marked ROOT
    for tok in doc:
        if tok.dep_ == "ROOT":
            return tok
    return None

root_node = find_root(parsed_doc)
print("root_node:", root_node.text)

root_node: discovered


## 3. Extract the Graph TerminalsFind the `nsubj` child of the root (`start_node`) and the token `"mutation"` (`end_node`).

In [39]:
def get_terminals(root, doc):
    start_node = next((c for c in root.children if c.dep_ == "nsubj"), None)
    end_node = next((t for t in doc if t.text == "mutation"), None)
    return start_node, end_node

start_node, end_node = get_terminals(root_node, parsed_doc)
print("start_node:", start_node.text)
print("end_node:", end_node.text)

start_node: researchers
end_node: mutation


## 4. Graph Traversal — Shortest PathTreat the parse as an undirected graph (edges go to `.head` and to every child) and BFS from `start_node` to `end_node`.

In [40]:
from collections import deque

def bfs_shortest_path(start, end):
    if start is None or end is None:
        return []
    if start.i == end.i:
        return [start]
    seen = {start.i}
    queue = deque([[start]])
    while queue:
        path = queue.popleft()
        node = path[-1]
        # every neighbour is either a child, or the head (skip self-loop on ROOT)
        neighbours = list(node.children)
        if node.head.i != node.i:
            neighbours.append(node.head)
        for nxt in neighbours:
            if nxt.i in seen:
                continue
            seen.add(nxt.i)
            new_path = path + [nxt]
            if nxt.i == end.i:
                return new_path
            queue.append(new_path)
    return []

dependency_path = bfs_shortest_path(start_node, end_node)
print("dependency_path:", " -> ".join(t.text for t in dependency_path))

dependency_path: researchers -> discovered -> vaccine -> prevents -> mutation


## 5. Path AnalysisPrint the syntactic details for just the tokens on `dependency_path`.

In [41]:
def print_path_analysis(path):
    print(f"{'Word':<15}{'POS':<8}{'Head':<15}{'Dependency':<10}")
    print("-" * 48)
    for tok in path:
        print(f"{tok.text:<15}{tok.pos_:<8}{tok.head.text:<15}{tok.dep_:<10}")

print_path_analysis(dependency_path)

Word           POS     Head           Dependency
------------------------------------------------
researchers    NOUN    discovered     nsubj     
discovered     VERB    discovered     ROOT      
vaccine        NOUN    discovered     dobj      
prevents       VERB    vaccine        relcl     
mutation       NOUN    prevents       dobj      


## 6. Visual VerificationRender `parsed_doc` with displaCy to eyeball that the table above matches the tree.

In [42]:
from spacy import displacy# renders inline in Jupyter/Colabdisplacy.render(parsed_doc, style="dep", jupyter=True, options={"distance": 100})